In [1]:
import os
%pwd

'c:\\Eye-Disease-Prediction\\research'

In [2]:
os.chdir("../")
%pwd

'c:\\Eye-Disease-Prediction'

In [3]:
import dagshub
dagshub.init(repo_owner='Satyam6024', repo_name='Eye-Disease-Prediction', mlflow=True)
os.environ["MLflow_Tracking_URI"] = "https://dagshub.com/Satyam6024/Eye-Disease-Prediction.mlflow"

Accessing as Satyam6024

Initialized MLflow to track repo "Satyam6024/Eye-Disease-Prediction"

Repository Satyam6024/Eye-Disease-Prediction initialized!

In [4]:
import tensorflow as tf

In [5]:
model = tf.keras.models.load_model("artifacts/training/model.h5")

In [6]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class EvaluationConfig:
    path_of_model: Path
    test_data: Path
    all_params: dict
    mlflow_uri: str
    params_image_size: list
    params_batch_size: int

In [7]:
from EyeDiseaseClassifier.constants import *
from EyeDiseaseClassifier.utils.common import read_yaml, create_directories, save_json

In [8]:
class ConfigurationManager:
    def __init__(
        self, 
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])
        
    def get_evaluation_config(self) -> EvaluationConfig:
        eval_config = EvaluationConfig(
            path_of_model="artifacts/training/model.h5",
            test_data="artifacts/data_ingestion/test",
            mlflow_uri="https://dagshub.com/Satyam6024/Eye-Disease-Prediction.mlflow",
            all_params=self.params,
            params_image_size=self.params.IMAGE_SIZE,
            params_batch_size=self.params.BATCH_SIZE
        )
        return eval_config

In [9]:
import tensorflow as tf
from pathlib import Path
import mlflow
import mlflow.keras
from urllib.parse import urlparse

In [ ]:
class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config
    
    
    def _valid_generator(self):
    
        datagenerator_kwargs = dict(
            rescale = 1./255
        )
    
        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            class_mode="categorical",
            interpolation="bilinear"
        )
    
        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )
    
        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.test_data,
            shuffle=False,
            **dataflow_kwargs
        )
    
    
    @staticmethod
    def load_model(path: Path) -> tf.keras.Model:
        return tf.keras.models.load_model(path)
    
    
    def evaluation(self):
        self.model = self.load_model(self.config.path_of_model)
        self._valid_generator()
        self.score = self.model.evaluate(self.valid_generator)
        self.save_score()
    
    def save_score(self):
        scores = {"loss": self.score[0], "accuracy": self.score[1]}
        save_json(path=Path("scores.json"), data=scores)
    
    
    def log_into_mlflow(self):
        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme
        
        with mlflow.start_run():
            mlflow.log_params(self.config.all_params)
            mlflow.log_metrics(
                {"loss": self.score[0], "accuracy": self.score[1]}
            )
            if tracking_url_type_store != "file":
                mlflow.keras.log_model(self.model, "model", registered_model_name="VGG16-Model")
            else:
                mlflow.keras.log_model(self.model, "model")

In [12]:
try:
    config = ConfigurationManager()
    eval_config = config.get_evaluation_config()
    evaluation = Evaluation(eval_config)
    evaluation.evaluation()
    evaluation.log_into_mlflow()

except Exception as e:
    raise e

[2026-08-06 13:48:53,148: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-08-06 13:48:53,151: INFO: common: yaml file: params.yaml loaded successfully]
[2026-08-06 13:48:53,153: INFO: common: created directory at: artifacts]
Found 10933 images belonging to 4 classes.
684/684 [==============================] - 161s 226ms/step - loss: 43.4621 - accuracy: 0.7989
[2026-08-06 13:51:38,062: INFO: common: json file saved at: scores.json]


2026/08/06 13:51:40 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


[2026-08-06 13:51:59,601: WARNING: save: Found untraced functions such as _jit_compiled_convolution_op, _jit_compiled_convolution_op, _jit_compiled_convolution_op, _jit_compiled_convolution_op, _jit_compiled_convolution_op while saving (showing 5 of 94). These functions will not be directly callable after loading.]
INFO:tensorflow:Assets written to: C:\Users\ssssa\AppData\Local\Temp\tmp9rknoo_2\model\data\model\assets
[2026-08-06 13:52:06,179: INFO: builder_impl: Assets written to: C:\Users\ssssa\AppData\Local\Temp\tmp9rknoo_2\model\data\model\assets]


c:\Users\ssssa\anaconda3\envs\tensorflow_env\lib\site-packages\_distutils_hack\__init__.py:26: UserWarning: Setuptools is replacing distutils.
  warnings.warn("Setuptools is replacing distutils.")
Successfully registered model 'InceptionV3Model'.
2026/08/06 13:53:14 INFO mlflow.tracking._model_registry.client: Waiting up to 300 seconds for model version to finish creation.                     Model name: InceptionV3Model, version 1
Created version '1' of model 'InceptionV3Model'.
